In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        library(data.table), 

        library(lme4), 
        library(emmeans), 
        
        # Plotting 
        library(patchwork), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Import 

In [ ]:
so <- readRDS("data/object/pp.rds")

# Cell count check 

In [ ]:
# Total cell genes vs cells 
nrow(so)
ncol(so)

# Genotype checks 

In [ ]:
# Genotype statistics 
vireo <- read.csv("result/genotype/genotype.csv", row.names=1)
vireo <- vireo[colnames(so), ]
vireo <- vireo[vireo$vireo_patient_id!="Patient_2", ]

In [ ]:
100*table(vireo$vireo_genotype_class)/sum(table(vireo$vireo_genotype_class))

In [ ]:
# Genotype time points  
vireo_1 <- vireo[colnames(so[, so$time_point %in% c("Baseline", "Tx")]), ]
table(vireo_1$vireo_genotype_best_class)
sum(table(vireo_1$vireo_genotype_best_class))

In [ ]:
# Genotype non-hematopoetic cells  
vireo_2 <- vireo[colnames(so[, so$celltype_domain %in% c("'Non-hematopoetic'")]), ]
table(vireo_2$vireo_genotype_best_class)
sum(table(vireo_2$vireo_genotype_best_class))

In [ ]:
# Celltype low genotype counts skin 
mat <- so@meta.data %>% dplyr::filter(hto_demux_class=="Skin") %>%
    dplyr::group_by(patient_id, time_point, celltype_low, genotype_class) %>% dplyr::summarise(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(time_point=paste0(time_point, "_", genotype_class)) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=)) %>% 
    as.data.frame() 

mat <- split(mat, f=mat$celltype_low)

mat <- lapply(mat, function(x) {

    x <- x %>% 
        tidyr::complete(patient_id=levels(patient_id), time_point=levels(time_point)) %>%
        tidyr::pivot_wider(id_cols=patient_id, names_from=time_point, values_from=n)

    rownames(x) <- x$patient_id
    x <- x[paste0("Patient_", 1:12), c("patient_id", "Baseline_Host", "Tx_Host", "D14_Host", "D14_Donor", "D100_Host", "D100_Donor", "GVHD_Host", "GVHD_Donor")]

    return(x)
    
}
      )

names(mat) <- str_replace_all(names(mat) , "\\+", "plus") %>% make.names()
openxlsx::write.xlsx(mat, "result/statistics/celltype_low_genotype_count_skin.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
# Celltype low genotype counts skin 
mat <- so@meta.data %>% dplyr::filter(hto_demux_class=="Blood") %>%
    dplyr::group_by(patient_id, time_point, celltype_low, genotype_class) %>% dplyr::summarise(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(time_point=paste0(time_point, "_", genotype_class)) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=)) %>% 
    as.data.frame() 

mat <- split(mat, f=mat$celltype_low)

mat <- lapply(mat, function(x) {

    x <- x %>% 
        tidyr::complete(patient_id=levels(patient_id), time_point=levels(time_point)) %>%
        tidyr::pivot_wider(id_cols=patient_id, names_from=time_point, values_from=n)

    rownames(x) <- x$patient_id
    x <- x[paste0("Patient_", 1:12), c("patient_id", "Baseline_Host", "Tx_Host", "D14_Host", "D14_Donor", "D100_Host", "D100_Donor", "GVHD_Host", "GVHD_Donor")]

    return(x)
    
}
      )

names(mat) <- str_replace_all(names(mat) , "\\+", "plus") %>% make.names()
openxlsx::write.xlsx(mat, "result/statistics/celltype_low_genotype_count_blood.xlsx", colNames=TRUE, rowNames=TRUE)

# Statistics 

In [ ]:
data <- read.csv("result/statistics/tc17_time_point_data.csv", row.names=1) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=names(color$time_point))) %>% 
    dplyr::mutate(
        ratio_cells=n_cells/n_cells_total,
        arcsine_cells=asin(sqrt(n_cells/n_cells_total))
    )

In [ ]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Tc17hi"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_tc17hi.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Tc17lo"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_tc17lo.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
data <- read.csv("result/statistics/th17_time_point_data.csv", row.names=1) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=names(color$time_point))) %>% 
    dplyr::mutate(
        ratio_cells=n_cells/n_cells_total,
        arcsine_cells=asin(sqrt(n_cells/n_cells_total))
    )

In [ ]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Th17hi"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_th17hi.xlsx", colNames=TRUE, rowNames=TRUE)

In [ ]:
model <- lmer(ratio_cells~time_point + (1|patient_id), data=data %>% dplyr::filter(hto_demux_class=="Skin" & data$t_cell_subset=="Th17lo"))
openxlsx::write.xlsx(emmeans(model, pairwise~time_point, adjust="sidak"), "result/statistics/tc17_time_point_skin_th17lo.xlsx", colNames=TRUE, rowNames=TRUE)

# abTCR analysis

In [ ]:
tcr_id_blood <- so@meta.data %>% dplyr::filter(hto_demux_class=="Blood") %>% dplyr::pull(clone_id) %>% na.omit()
tcr_id_skin <- so@meta.data %>% dplyr::filter(hto_demux_class=="Skin") %>% dplyr::pull(clone_id) %>% na.omit()

In [ ]:
tcr_id_shared <- intersect(tcr_id_blood, tcr_id_skin)

In [ ]:
mat_1 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared) %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

In [ ]:
mat_2 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared & genotype_class=="Host") %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

In [ ]:
mat_3 <- so@meta.data %>% dplyr::filter(clone_id %in% tcr_id_shared & genotype_class=="Donor") %>% dplyr::select(patient_id, clone_id) %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(n_tcr=length(unique(clone_id)), n_cells_tcr_total=n()) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::mutate(n_cells_tcr=n()) %>% 
    dplyr::group_by(patient_id, n_tcr, n_cells_tcr_total) %>% dplyr::summarise(n_cells_tcr_mean=mean(n_cells_tcr))

In [ ]:
openxlsx::write.xlsx(list(all=mat_1, host=mat_2, donor=mat_2), "result/statistics/shared_tcr_blood_skin.xlsx", colNames=TRUE, rowNames=TRUE)